In [ ]:
import io
import re
import time
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import requests

# ============================================================
# BUILD: sp500_top10_semiannual_2007_2026_ivv_proxy.csv
# Source: iShares IVV holdings CSV snapshots (proxy for S&P 500 weights)
# Output schema used by your backtester:
#   snapshot_date, rank, ticker, weight_pct
# Also writes a "wide" helper file.
# ============================================================

START_DATE = pd.Timestamp("2007-05-15")
END_DATE = pd.Timestamp.today().normalize()  # or set explicit date
OUT_LONG = "sp500_top10_semiannual_2007_2026_ivv_proxy.csv"
OUT_WIDE = "sp500_top10_semiannual_2007_2026_ivv_proxy_wide.csv"

IVV_PRODUCT_IDS = ["239726"]
IVV_PATHS = [
    "us/products/{pid}/ishares-core-sp-500-etf/1467271812596.ajax",
    "us/individual/products/{pid}/ishares-core-sp-500-etf/1467271812596.ajax",
    "us/financial-professionals/products/{pid}/ishares-core-sp-500-etf/1467271812596.ajax",
]
IVV_HOSTS = [
    "https://www.ishares.com",
    "https://www.blackrock.com",
]

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept": "text/csv,text/plain,*/*",
    "Referer": "https://www.ishares.com/us/products/239726/ishares-core-sp-500-etf",
}

SESSION = requests.Session()
SESSION.headers.update(HEADERS)


def normalize_ticker_for_yf(t: str):
    if pd.isna(t):
        return None
    t = str(t).strip().upper().replace(" ", "")
    t = t.replace("/", "-").rstrip("*").replace(".", "-")
    mapping = {
        "BRKB": "BRK-B",
        "BRK-B": "BRK-B",
        "BRK.B": "BRK-B",
        "FB": "META",
        "GOOGL": "GOOGL",
        "GOOG": "GOOG",
        "BFB": "BF-B",
        "BF.B": "BF-B",
    }
    return mapping.get(t, t)


def _candidate_urls_and_params(asof: pd.Timestamp):
    # Try multiple param spellings and date formats because BlackRock/iShares endpoints vary.
    date_variants = [asof.strftime("%Y%m%d"), asof.strftime("%Y-%m-%d")]
    param_keys = ["asOfDate", "asofDate", "asofdate"]

    for host in IVV_HOSTS:
        for path_tmpl in IVV_PATHS:
            for pid in IVV_PRODUCT_IDS:
                base = f"{host}/" + path_tmpl.format(pid=pid)
                for dv in date_variants:
                    for pk in param_keys:
                        params = {
                            "fileType": "csv",
                            "fileName": "IVV_holdings",
                            "dataType": "fund",
                            pk: dv,
                        }
                        yield base, params


def _extract_snapshot_date_from_text(raw_text: str, fallback: pd.Timestamp) -> pd.Timestamp:
    # Common iShares CSV metadata patterns seen historically.
    patterns = [
        r"Fund Holdings as of,\s*([0-9]{2}/[0-9]{2}/[0-9]{4})",
        r"As of Date,\s*([0-9]{2}/[0-9]{2}/[0-9]{4})",
        r"as of\s*([A-Za-z]{3,9}\s+[0-9]{1,2},\s+[0-9]{4})",
    ]
    for pat in patterns:
        m = re.search(pat, raw_text, flags=re.IGNORECASE)
        if m:
            dt = pd.to_datetime(m.group(1), errors="coerce")
            if pd.notna(dt):
                return pd.Timestamp(dt).normalize()
    return pd.Timestamp(fallback).normalize()


def _parse_ishares_holdings_csv(raw_text: str, requested_asof: pd.Timestamp) -> tuple[pd.DataFrame, pd.Timestamp]:
    lines = raw_text.splitlines()

    # Find the actual holdings header row.
    header_idx = None
    for i, line in enumerate(lines):
        if line.startswith("Ticker,") and ("Weight" in line or "Weight (%)" in line):
            header_idx = i
            break
    if header_idx is None:
        raise ValueError("Could not locate holdings CSV header row ('Ticker,...,Weight').")

    snapshot_date = _extract_snapshot_date_from_text(raw_text, requested_asof)

    body = "\n".join(lines[header_idx:])
    df = pd.read_csv(io.StringIO(body))
    df.columns = [str(c).strip() for c in df.columns]

    # Standardize likely column names.
    colmap = {c.lower(): c for c in df.columns}

    ticker_col = next((colmap[k] for k in colmap if k == "ticker"), None)
    weight_col = None
    for k in colmap:
        if "weight" in k and "%" in k:
            weight_col = colmap[k]
            break
    if weight_col is None:
        for k in colmap:
            if "weight" in k:
                weight_col = colmap[k]
                break

    name_col = next((colmap[k] for k in colmap if k in {"name", "issuer name", "security name"}), None)
    asset_class_col = next((colmap[k] for k in colmap if "asset class" in k), None)

    if ticker_col is None or weight_col is None:
        raise ValueError(f"Missing required columns in holdings CSV. Columns found: {list(df.columns)}")

    # Clean rows: drop footer/blank rows.
    out = df.copy()
    out[ticker_col] = out[ticker_col].astype(str).str.strip()
    out[weight_col] = pd.to_numeric(out[weight_col], errors="coerce")
    out = out.dropna(subset=[weight_col])
    out = out[out[weight_col] > 0]
    out = out[~out[ticker_col].isin(["", "-", "N/A", "CASH", "USD"])]

    if asset_class_col is not None:
        # Keep equities; if column missing or all-null, proceed without filter.
        mask = out[asset_class_col].astype(str).str.contains("equity", case=False, na=False)
        if mask.any():
            out = out[mask]

    out = out.assign(
        ticker=out[ticker_col].map(normalize_ticker_for_yf),
        weight_pct=out[weight_col].astype(float),
        company_name=(out[name_col].astype(str).str.strip() if name_col is not None else np.nan),
    )
    out = out.dropna(subset=["ticker", "weight_pct"])

    # Remove duplicates if any, keeping highest weight version.
    out = out.sort_values(["ticker", "weight_pct"], ascending=[True, False]).drop_duplicates("ticker", keep="first")
    out = out.sort_values("weight_pct", ascending=False).head(10).reset_index(drop=True)
    out["rank"] = np.arange(1, len(out) + 1)
    out["snapshot_date"] = snapshot_date

    if out.empty:
        raise ValueError("Parsed holdings file but no valid top-10 rows remained.")

    return out[["snapshot_date", "rank", "ticker", "weight_pct", "company_name"]], snapshot_date


def fetch_ivv_top10_for_date(target_date: pd.Timestamp, max_offset_days: int = 7, sleep_sec: float = 0.2):
    # Search nearest available holdings snapshot around target date.
    offsets = [0]
    for k in range(1, max_offset_days + 1):
        offsets.extend([k, -k])

    errors = []
    for off in offsets:
        asof = pd.Timestamp(target_date) + pd.Timedelta(days=off)
        for url, params in _candidate_urls_and_params(asof):
            try:
                r = SESSION.get(url, params=params, timeout=25)
                txt = r.text or ""
                if r.status_code != 200:
                    errors.append(f"{asof.date()} {r.status_code} @ {url}")
                    continue
                # Quick screen for non-CSV responses.
                if "Ticker," not in txt or "Weight" not in txt:
                    errors.append(f"{asof.date()} non-holdings payload @ {url}")
                    continue
                top10, actual_snapshot = _parse_ishares_holdings_csv(txt, asof)
                top10["target_date"] = pd.Timestamp(target_date).normalize()
                top10["fetched_asof_param"] = asof.normalize()
                top10["source_url"] = url
                if sleep_sec:
                    time.sleep(sleep_sec)
                return top10, actual_snapshot
            except Exception as e:
                errors.append(f"{asof.date()} exception: {type(e).__name__}: {e}")
                continue

    # Trim logged errors for readability
    err_preview = "\n".join(errors[:10])
    raise RuntimeError(
        f"Could not fetch IVV holdings near {pd.Timestamp(target_date).date()} (±{max_offset_days}d). "
        f"Sample attempts:\n{err_preview}"
    )


def build_target_dates(start: pd.Timestamp, end: pd.Timestamp) -> list[pd.Timestamp]:
    start = pd.Timestamp(start).normalize()
    end = pd.Timestamp(end).normalize()
    dates = []
    for y in range(start.year, end.year + 1):
        for m, d in [(6, 30), (12, 31)]:
            dt = pd.Timestamp(year=y, month=m, day=d)
            if start <= dt <= end:
                dates.append(dt)

    # Add latest snapshot target (today) if not already covered.
    if not dates or dates[-1] < end:
        dates.append(end)

    # Ensure unique sorted timestamps
    dates = sorted(pd.DatetimeIndex(dates).normalize().unique())
    return [pd.Timestamp(d) for d in dates]


def main():
    targets = build_target_dates(START_DATE, END_DATE)
    all_rows = []

    print(f"Target snapshot dates: {len(targets)}")
    for i, t in enumerate(targets, 1):
        print(f"[{i}/{len(targets)}] Fetching around {t.date()} ...", end=" ")
        top10, actual = fetch_ivv_top10_for_date(t)
        print(f"OK -> actual {actual.date()} | {len(top10)} rows")
        all_rows.append(top10)

    long_df = pd.concat(all_rows, ignore_index=True)

    # Dedupe on actual snapshot_date + rank if a target resolves to same snapshot as neighbor.
    long_df["snapshot_date"] = pd.to_datetime(long_df["snapshot_date"]).dt.normalize()
    long_df = (long_df
               .sort_values(["snapshot_date", "rank", "weight_pct"], ascending=[True, True, False])
               .drop_duplicates(subset=["snapshot_date", "rank"], keep="first")
               .copy())

    # Keep only core columns required by the backtester, plus optional metadata.
    long_df["weight_pct"] = long_df["weight_pct"].astype(float)
    long_df["weight_norm_top10"] = long_df.groupby("snapshot_date")["weight_pct"].transform(lambda s: s / s.sum())

    long_out = long_df[[
        "snapshot_date", "rank", "ticker", "weight_pct",
        "weight_norm_top10", "company_name", "target_date", "fetched_asof_param"
    ]].sort_values(["snapshot_date", "rank"]).reset_index(drop=True)

    # Wide helper file: one row per snapshot, ticker_i and weight_pct_i columns.
    wide_parts = []
    for k in range(1, 11):
        sub = long_out[long_out["rank"] == k][["snapshot_date", "ticker", "weight_pct"]].copy()
        sub = sub.rename(columns={"ticker": f"ticker_{k}", "weight_pct": f"weight_pct_{k}"})
        wide_parts.append(sub)

    wide_df = None
    for part in wide_parts:
        wide_df = part if wide_df is None else wide_df.merge(part, on="snapshot_date", how="outer")
    wide_df = wide_df.sort_values("snapshot_date").reset_index(drop=True)

    long_out.to_csv(OUT_LONG, index=False)
    wide_df.to_csv(OUT_WIDE, index=False)

    print("\nSaved:")
    print(f" - {OUT_LONG}")
    print(f" - {OUT_WIDE}")
    print(f"Rows (long): {len(long_out)}")
    print("\nPreview:")
    print(long_out.head(12).to_string(index=False))


if __name__ == "__main__":
    main()
